In [ ]:
import pandas as pd

# Increase training subset from thesis to include ~2000 slides
# Original, 100 samples x 10 tissue = Data\abmil_exp3
# Original, 10 samples x 10 tissue, not in train = Data\abmil_inference_exp3
# Here, sample up to 300, avoid slides from abmil_exp3: D:\DATA\abmil_training_vers3.csv

# File paths
abmil_path = r"D:\DATA\abmil_exp3.csv" # Previous smaller subset (100x10 = 1000)
all_path = r"D:\DATA\with_snomed_category.csv" # All WSIs

# Load datasets
df_abmil = pd.read_csv(abmil_path)
df_all = pd.read_csv(all_path)

In [ ]:
from helper_functions import strings2lists, lists2tuples

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)
    df_abmil[col] = df_abmil[col].apply(strings2lists)

df_abmil = lists2tuples(df_abmil)
df_all = lists2tuples(df_all)

In [ ]:
# Check present T_category values in abmil_exp3
present_categories = df_abmil[df_abmil["T_category"].apply(lambda x: len(x) == 1)]["T_category"].unique()

print("T_categories in abmil_exp3:")
for cat in present_categories:
    print(cat)

In [ ]:
# Identify unique slide IDs already in abmil_exp3
existing_rekvnr = set(df_abmil["rekvnr"].unique())
print(len(existing_rekvnr))

In [ ]:
from helper_functions import subset_df, subset_df_list

# Exclude slides used for training in abmil_exp3
filtered_df = df_all[(~df_all["rekvnr"].isin(existing_rekvnr))]
print("Without rekvnr used for training: ", len(filtered_df))

# Apply HE + Hist. store filters on filtered dataset
df_HE = subset_df(filtered_df, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")

sampled_dfs = []

for cat in present_categories: 
    if isinstance(cat, tuple):
        cat = cat[0]
    cat = str(cat).strip()

    # All slides matching this category
    df_cat = subset_df_list(df_HE, "T_category", cat)

    # Prioritize single T category slides
    single_cat = df_cat[df_cat["T_category"].apply(lambda x: len(x) == 1)]

    # Remaining multi-category slides
    multi_cat = df_cat.drop(single_cat.index)

    # Sample up to 20 slides following the priority order
    remaining = 300
    sampled_cat = pd.DataFrame()
    groups = [single_cat, multi_cat]
    for grp in groups:
        if remaining <= 0:
            break
        if len(grp) > 0:
            take = min(remaining, len(grp))
            sampled_part = grp.sample(n=take, random_state=42)
            sampled_cat = pd.concat([sampled_cat, sampled_part], ignore_index=True)
            remaining -= take
    sampled_dfs.append(sampled_cat)

df_selected = pd.concat(sampled_dfs, ignore_index=True)
df_selected = df_selected.drop_duplicates(subset="filename")

print("Final selected slides:", len(df_selected))
print("\nSlides per category:")
print(df_selected["T_category"].value_counts())

In [ ]:
# Save to csv
output_file = r"D:\DATA\abmil_training_vers3.csv"
df_selected.to_csv(output_file, index=False)

print(f"Saved DataFrame to {output_file}")